# पाठ १२ - एजेन्ट स्क्र्याचप्याडसँग चैट इतिहास कमी

यो नोटबुकले Microsoft Agent Framework प्रयोग गरेर लामो कुराकानीमा सन्दर्भ कसरी व्यवस्थापन गर्ने देखाउँछ। कुराकानी बढ्दै जाँदा, टोकन गणना बढ्छ — अन्ततः मोडलको सन्दर्भ विन्डो भन्दा बढी हुन्छ। हामी यसलाई **सन्दर्भ सारांशिकरण नमूना** र **एजेन्ट स्क्र्याचप्याड** को प्रयोग गरेर स्थायी स्मृतिका लागि समाधान गर्छौं।

## तपाईंले के सिक्नुहुनेछ:
१. **किन सन्दर्भ व्यवस्थापन महत्त्वपूर्ण छ**: टोकन सीमाहरू र सन्दर्भ विन्डोहरूको बुझाइ
२. **सन्दर्भ-संश्लिष्ट एजेन्टहरू**: आफ्नै कुराकानी सन्दर्भ व्यवस्थापन गर्ने एजेन्टहरू बन्ने
३. **सन्दर्भ सारांशिकरण नमूना**: कुराकानी इतिहासलाई सङ्क्षेप गर्न उपकरणहरू प्रयोग गर्ने
४. **एजेन्ट स्क्र्याचप्याड**: सन्दर्भ कमी हुँदा पनि जीवित रहने स्थायी स्मृति

## पूर्वआवश्यकताहरू:
- वातावरण चरहरू कन्फिगर गरिएको Azure OpenAI सेटअप
- पहिलाका पाठहरूबाट आधारभूत एजेन्ट अवधारणाहरूको बुझाइ


## सेटअप


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

In [ ]:
import os
import asyncio
import dotenv
from datetime import datetime
from pathlib import Path

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

In [ ]:
dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## किन सन्दर्भ व्यवस्थापन महत्वपूर्ण छ

प्रत्येक LLM को एउटा सीमित **सन्दर्भ विन्डो** हुन्छ — यसले एकल अनुरोधमा प्रशोधन गर्न सक्ने अधिकतम टोकनको संख्या। बहु-चरण संवाद बढ्दै गएसँगै:

- **प्रत्येक प्रयोगकर्ता सन्देश र सहायक जवाफसंग टोकन संख्या रेखीय रूपमा बढ्छ**।
- **प्रॉम्प्ट टोकनहरूले लागतलाई प्रभाब पार्छ** किनभने सम्पूर्ण इतिहास हरेक पटक पुन: पठाइन्छ।
- अन्ततः संवादले **सन्दर्भ विन्डो पार गर्छ** र मोडेलले वा त कटौती गर्छ वा त्रुटि देखाउँछ।

### सन्दर्भ व्यवस्थापनका रणनीतिहरू

| रणनीति | यसले कसरी काम गर्छ | सट्टा |
|---|---|---|
| **कटौती** | पुराना सन्देशहरू हटाउने | प्रारम्भिक सन्दर्भ हराउँछ |
| **सारांश** | पुराना सन्देशहरूलाई सारांशमा संक्षिप्त गर्ने | केही विवरण हराउँछ, तर मुख्य बुँदा राखिन्छ |
| **स्क्र्याचप्याड / बाह्य स्मृति** | संवादको बाहिर मुख्य तथ्यहरू भण्डारण गर्ने | उपकरण कल आवश्यक पर्छ, तर कुनै पनि कटौतीबाट बच्छ |

यस नोटबुकमा हामी **सारांश** र **स्क्र्याचप्याड उपकरण** संयोजन गर्छौं जसले एजेन्टलाई संवाद इतिहास संक्षिप्त गरिएको अवस्थामा पनि निरन्तरता कायम गर्न सक्षम बनाउँछ।


## सन्दर्भ-चेतन एजेन्ट सिर्जना गर्दै


In [ ]:
agent = client.as_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("Context-aware travel planning agent created")

## लामो कुराकानीको अनुकरण

सन्दर्भ कसरी जम्मा हुन्छ भन्ने हेर्न हामी बहु-पटकको कुराकानी मार्फत जान्छौं। एजेन्टले प्रत्येक चरणमा प्रमुख विवरणहरू (रुचिहरू, बजेट, यात्रा मितिहरू) राख्नुपर्छ र निरन्तरता देखाउनुपर्छ।


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

एजेन्टले कसरी पहिलेका संवादहरूबाट सन्दर्भ राख्छ भनेर ध्यान दिनुहोस् — यसले जापान, सुशी, मन्दिरहरू, फोटोग्राफी, $३००० बजेट, एकल यात्रा, र अप्रिलको समयसीमा को थाहा छ। छोटो कुराकानीमा यो राम्रो काम गर्छ, तर जति कुराकानी बढ्छ, पुरा इतिहास पुनः पठाउनु महँगो हुन्छ।

सन्दर्भ संचय देख्न थप संवादहरूका साथ कुराकानी जारी राखौं:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## सन्दर्भ संक्षेपण ढाँचा

वार्तालाप बढ्दै जादा, हामीले सङ्ग्रहित सन्दर्भलाई सङ्कुचित स्वरूपमा रूपान्तरण गर्न **संक्षेपण उपकरण** प्रयोग गर्न सक्छौं। एजेन्टले यस उपकरणलाई प्रमुख प्राथमिकताहरू रेकर्ड गर्न कल गर्दछ ताकि पुराना सन्देशहरू हटाइए पनि आवश्यक जानकारी सुरक्षित रहोस्।

यो ढाँचा थप परिष्कृत इतिहास कमीकरणको लागि आधारभूत ब्लक हो:
1. एजेन्टले वार्तालापबाट प्रमुख तथ्यहरू पहिचान गर्दछ
2. यसले तिनीहरूलाई कायम राख्न संक्षेपण उपकरणलाई कल गर्दछ
3. पुराना सन्देशहरू सुरक्षित तरिकाले हटाउन सकिन्छ किनकि संक्षेपले के महत्वपूर्ण छ समेट्छ

तल हामी `summarize_preferences` नामक उपकरण परिभाषित गर्छौं जुन एजेन्टले यसले सिकेको कुराको सङ्कुचित सारांश रेकर्ड गर्न कल गर्न सक्छ।


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = client.as_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## सारांश

यस पाठमा तपाईंले Microsoft Agent Framework प्रयोग गरेर लामो अवधि चल्ने एजेन्ट कुराकानीहरूमा सन्दर्भ कसरी व्यवस्थापन गर्ने सिक्नुभयो:

### मुख्य अवधारणाहरू
- **सन्दर्भ विन्डोज सीमित हुन्छन्** — कुराकानी इतिहासमा प्रत्येक टोकनले पैसा लाग्छ र सीमा तर्फ गणना हुन्छ।
- **सारांश उपकरणहरू** एजेन्टलाई सञ्चित सन्दर्भलाई सङ्कुचित सारांशमा रूपान्तरण गर्न दिन्छन्, झन्डै टोकन प्रयोग घटाउँछन् र आवश्यक जानकारी जोगाउँछन्।
- **एजेन्ट स्क्र्याचप्याडहरू** दीगो बाह्य स्मृति प्रदान गर्छन् जुन कुनै पनि कुराकानी सानो बनाइएपनि बाँच्छ।

### तपाईंले के बनाउनु भयो
- एउटा **सन्दर्भ-सावधान एजेन्ट** जसले बहु-चरण कुराकानीहरूमा निरन्तरता कायम राख्छ
- एउटा **सारांश उपकरण** (`summarize_preferences`) जसले मुख्य प्रयोगकर्ता विवरण सङ्कुचित ढाँचामा रेकर्ड गर्छ
- एउटा **बहु-चरण कुराकानी** जसले सन्दर्भ कायम राख्ने र परिवर्तन ह्यान्डल गर्ने देखाउँछ

### वास्तविक दुनियाँका प्रयोगहरू
- **ग्राहक सेवा बोटहरू**: लामो समर्थन सत्रहरूमा प्राथमिकताहरू सम्झन्छन्
- **व्यक्तिगत सहायकहरू**: सन्दर्भ पुनः व्याख्या नगरिकन चलिरहेका परियोजनाहरू ट्र्याक गर्छन्
- **शैक्षिक ट्युटरहरू**: धेरै अन्तरक्रियाहरूमा विद्यार्थीको प्रगति कायम राख्छन्

### आगामी कदमहरू
- फाइल-आधारित दीगोता सहित पूर्ण स्क्र्याचप्याड उपकरण लागू गर्नुहोस्
- सारांशपछि स्वचालित इतिहास छोट्याउने थप्नुहोस्
- अर्थपूर्ण स्मृति खोजका लागि भेक्टर डेटाबेससहित संयोजन गर्नुहोस्
- पूर्ण सन्दर्भसहित कुराकानीहरू दिनहरू पछि पुनः सुरु गर्न सक्ने एजेन्टहरू बनाउनुहोस्


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
यो दस्तावेज़ AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) प्रयोग गरेर अनुवाद गरिएको हो। हामी सही हुन प्रयास गर्छौं, तर कृपया जानकार हुनुस् कि स्वचालित अनुवादमा त्रुटिहरू वा अशुद्धताहरू हुन सक्छन्। मूल दस्तावेज़ यसको मूल भाषामा आधिकारिक स्रोत मानिनुपर्छ। महत्वपूर्ण जानकारीका लागि व्यावसायिक मानव अनुवाद सिफारिस गरिन्छ। यस अनुवादको प्रयोगबाट उत्पन्न कुनै पनि गलत बुझाइ वा त्रुटिको लागि हामी जिम्मेवार छैनौं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
